# 09 - Comparacion de Modelos

## Objetivo
Comparar todos los modelos entrenados en metricas tecnicas y de negocio.

## Modelos
| Modelo | Framework | Tarea |
|--------|-----------|-------|
| ARIMA | statsmodels | Prediccion univariada |
| Random Forest | sklearn | Clasificacion 4 clases |
| BiLSTM+Attention | PyTorch | Prediccion binaria 24h |
| Autoencoder | PyTorch | Deteccion anomalias |
| CNN 1D | PyTorch | Clasificacion 4 clases |
| TF LSTM | TensorFlow | Prediccion binaria 24h |

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.preprocessor import FiberDataPreprocessor
from src.data.feature_engineer import FiberFeatureEngineer
from src.data.dataset import create_dataloaders
from src.models.lstm_predictor import LSTMPredictor
from src.models.cnn1d_classifier import CNN1DClassifier
from src.models.autoencoder import FiberAutoencoder
from src.models.baseline import RandomForestBaseline
from src.training.evaluator import ModelEvaluator
from src.utils.metrics import calculate_business_metrics, format_business_report
from src.utils.config_loader import load_config, get_path
from src.utils.visualization import apply_style, plot_model_comparison

apply_style()
config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Preparar datos
preprocessor = FiberDataPreprocessor()
train_df, val_df, test_df = preprocessor.process_pipeline()

feature_cols = ['optical_power_dbm', 'attenuation_db_km', 'ber', 'osnr_db',
                'chromatic_dispersion', 'temperature', 'humidity', 'log_ber']
feature_cols = [c for c in feature_cols if c in train_df.columns]

seq_len = config['preprocessing']['sequence_length']
evaluator = ModelEvaluator(device)

In [ ]:
# Cargar y evaluar cada modelo
all_results = {}

# 1. BiLSTM (Prediccion binaria)
print('=== BiLSTM + Attention ===')
try:
    _, _, test_loader_binary = create_dataloaders(
        train_df, val_df, test_df, feature_cols, 'fault_in_24h',
        seq_len, config['models']['lstm_predictor']['batch_size'], 'sequence'
    )
    lstm_model = LSTMPredictor(len(feature_cols), **{k: v for k, v in config['models']['lstm_predictor'].items() if k in ['hidden_size', 'num_layers', 'dropout', 'bidirectional']}).to(device)
    lstm_model.load_state_dict(torch.load(str(get_path('models') / 'lstm_best.pt'), weights_only=True, map_location=device))
    all_results['BiLSTM+Attn'] = evaluator.evaluate_binary(lstm_model, test_loader_binary)
except Exception as e:
    print(f'LSTM no disponible: {e}')
    all_results['BiLSTM+Attn'] = {'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0, 'roc_auc': 0}

# 2. CNN 1D (Clasificacion multi-clase)
print('\n=== CNN 1D ===')
try:
    _, _, test_loader_multi = create_dataloaders(
        train_df, val_df, test_df, feature_cols, 'fault_label',
        seq_len, config['models']['cnn1d']['batch_size'], 'sequence'
    )
    cnn_model = CNN1DClassifier(len(feature_cols), **{k: v for k, v in config['models']['cnn1d'].items() if k in ['channels', 'kernel_size', 'num_classes', 'dropout']}).to(device)
    cnn_model.load_state_dict(torch.load(str(get_path('models') / 'cnn1d_best.pt'), weights_only=True, map_location=device))
    cnn_results = evaluator.evaluate_multiclass(cnn_model, test_loader_multi, ['normal', 'physical_cut', 'degradation', 'bad_splice'])
    all_results['CNN1D'] = cnn_results
except Exception as e:
    print(f'CNN1D no disponible: {e}')
    all_results['CNN1D'] = {'accuracy': 0, 'f1_weighted': 0, 'f1_macro': 0}

print('\nModelos evaluados:', list(all_results.keys()))

In [ ]:
# Tabla comparativa
print(ModelEvaluator.compare_models(all_results))

In [ ]:
# Grafico comparativo
comparison_metrics = {}
for name, metrics in all_results.items():
    comparison_metrics[name] = {
        'Accuracy': metrics.get('accuracy', 0),
        'F1': metrics.get('f1', metrics.get('f1_weighted', 0)),
        'AUC': metrics.get('roc_auc', 0),
    }

fig = plot_model_comparison(comparison_metrics, ['Accuracy', 'F1', 'AUC'])
plt.show()

In [ ]:
# Metricas de negocio (usando el mejor modelo binario)
print('\n=== METRICAS DE NEGOCIO ===')

# Generar predicciones del mejor modelo
best_model_name = max(
    [(k, v.get('f1', v.get('f1_weighted', 0))) for k, v in all_results.items()],
    key=lambda x: x[1]
)[0]
print(f'Mejor modelo: {best_model_name}')

# Simular predicciones para metricas de negocio
y_true = test_df['fault_in_24h'].values if 'fault_in_24h' in test_df.columns else test_df['is_fault'].values
best_metrics = all_results[best_model_name]
recall = best_metrics.get('recall', best_metrics.get('f1_weighted', 0.8))

# Simular predicciones basadas en el recall del modelo
np.random.seed(42)
y_pred = y_true.copy()
fault_idx = np.where(y_true == 1)[0]
n_miss = int(len(fault_idx) * (1 - recall))
if n_miss > 0:
    miss_idx = np.random.choice(fault_idx, size=n_miss, replace=False)
    y_pred[miss_idx] = 0

business_metrics = calculate_business_metrics(
    y_true, y_pred,
    pre_fault_hours=test_df['pre_fault_hours'].values if 'pre_fault_hours' in test_df.columns else None,
)
print(format_business_report(business_metrics))